
# Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet")\
    .option("header",True)\
    .option("inferSchema", True)\
    .load("abfss://bronze@databrikcsete.dfs.core.windows.net/orders")

In [0]:
display(df)

In [0]:
df = df.drop("_rescued_data")

In [0]:
display(df)

In [0]:
df = df.withColumn("order_date",to_timestamp(col("order_date")))

display(df)

In [0]:
df = df.withColumn("year",year(col("order_date")))

display(df)

In [0]:
from pyspark.sql.window import Window

In [0]:
df1 = df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))

display(df1)

# Data Writing

In [0]:
display(df)

In [0]:
df.write.format("delta")\
    .mode("append")\
    .option("path", "abfss://silver@databrikcsete.dfs.core.windows.net/orders")\
    .save()